In [ ]:
# autoreload modules when they change
%load_ext autoreload
%autoreload 2

In [ ]:
from datetime import datetime
from typing import Final

import matplotlib.pyplot as plt
import numpy as np

# package imports
from gdhprediction.apis import get_gdh_prediction
from gdhprediction.climate import ClimateDataRequest, ClimateGdhData
from gdhprediction.config import GdhLocalizedPredictionConfig, GdhPredictionConfig
from gdhprediction.utils import PERCENTILES

In [ ]:
ESSEN: Final[tuple[float, float]] = (51.448806, 7.019444)
cfg = GdhPredictionConfig(transform_key="radiation_shift")
cd: ClimateGdhData = ClimateGdhData.from_params(*ESSEN, cfg, start_year=2006, end_year=2025)
cdr = ClimateDataRequest(cd.request_data)
nasa_20 = cdr.request_nasa_data()
processed = cd._process(nasa_20)

In [ ]:
# get 30 day forecast
forecast = cd.forecast_gdh()

In [ ]:
# plot forecast for each percentile
plt.figure(figsize=(10, 6))
for p in PERCENTILES:
    plt.plot(forecast[f'GDH_p{p:02d}'], label=f'{p}th percentile')
plt.xlabel('Hour since today')
plt.ylabel('Growing Degree Hours (GDH)')
plt.title('28 Day Forecast of Growing Degree Hours')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# plot 5 percentiles of the GDH
plt.figure(figsize=(10, 6))
percentiles = [5, 25, 50, 75, 95]
for p in percentiles:
    plt.plot(processed[f'CUMGDH_p{p:02d}'], label=f'{p}th percentile')
plt.xlabel('Hour of Year')
plt.ylabel('Growing Degree Hours (GDH)')
plt.title('Cumulative Growing Degree Hours')
plt.legend()
plt.grid()
plt.show()

In [ ]:
lat, lon = ESSEN
loc_cfg = GdhLocalizedPredictionConfig.pack(lat, lon, cfg)
start = datetime.fromisoformat("2026-05-11T23:00:00+02:00")
data = get_gdh_prediction(loc_cfg, maybe_datetime=start)

In [ ]:
daily = get_gdh_prediction(loc_cfg, maybe_datetime=start, frequency='daily')

In [ ]:
# plot the forecast for hourly and daily data
plt.figure(figsize=(10, 6))
plt.plot(data, label='Hourly GDH')
# daily as markers
plt.scatter(np.linspace(24, 24 * 28, 28), daily, label='Daily GDH', color='orange', marker='o')
plt.xlabel('Hour since today')
plt.ylabel('Growing Degree Hours (GDH)')
plt.title('28 Day Forecast of Growing Degree Hours')
plt.legend()
plt.grid()
plt.show()